# Churn Prediction Model & Model Card

This notebook trains, evaluates, and selects a machine learning model to predict D2C customer churn within a 60-day window, optimizes the threshold from a business perspective, and conducts error analysis.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

DATA = Path('data')


## 1. Load Modeling Snapshot

In [2]:
df = pd.read_csv(DATA / "rfm_modeling_snapshot.csv")
print("Data shape:", df.shape)
print("Target distribution by split:")
print(df.groupby('split')['churn_next_60d'].agg(['count', 'mean']))


Data shape: (2400, 29)
Target distribution by split:
            count      mean
split                      
test          336  0.500000
train        1728  0.469907
validation    336  0.437500


## 2. Separate Features and Splits

In [3]:
y = df["churn_next_60d"]
drop_cols = ["customer_id", "snapshot_date", "churn_next_60d", "split"]
X = df.drop(columns=drop_cols)

# Separate categorical and numerical features
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

splits = {name: df["split"] == name for name in ["train", "validation", "test"]}


## 3. Build Preprocessing Pipeline

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ]
)


## 4. Train Models
Train Logistic Regression and Random Forest models.

In [5]:
models = {
    "baseline_logistic_regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "random_forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=350, min_samples_leaf=8, class_weight="balanced_subsample", random_state=42, n_jobs=-1))
    ])
}

# Fit models
for name, model in models.items():
    model.fit(X[splits["train"]], y[splits["train"]])
    print(f"Trained {name} successfully.")


Trained baseline_logistic_regression successfully.


Trained random_forest successfully.


## 5. Threshold Optimization & Selection
Maximize validation-set business utility where a prevented churner is worth INR 600, and outreach cost is INR 80.

In [6]:
def evaluate_model(y_true, y_proba, threshold):
    preds = (y_proba >= threshold).astype(int)
    return {
        "threshold": round(float(threshold), 4),
        "accuracy": round(float(accuracy_score(y_true, preds)), 4),
        "precision": round(float(precision_score(y_true, preds, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, preds, zero_division=0)), 4),
        "f1": round(float(f1_score(y_true, preds, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_true, y_proba)), 4),
        "pr_auc": round(float(average_precision_score(y_true, y_proba)), 4),
        "confusion_matrix": confusion_matrix(y_true, preds).tolist()
    }

# Check validation performance
results = {}
for name, model in models.items():
    val_proba = model.predict_proba(X[splits["validation"]])[:, 1]
    test_proba = model.predict_proba(X[splits["test"]])[:, 1]
    results[name] = {
        "validation_default_0.50": evaluate_model(y[splits["validation"]], val_proba, 0.5),
        "test_default_0.50": evaluate_model(y[splits["test"]], test_proba, 0.5)
    }

# Select best model based on validation PR AUC
best_name = max(models.keys(), key=lambda n: results[n]["validation_default_0.50"]["pr_auc"])
best_model = models[best_name]
print(f"Selected model: {best_name}")

# Optimize threshold based on business utility
val_y = y[splits["validation"]]
val_proba = best_model.predict_proba(X[splits["validation"]])[:, 1]
thresholds = np.arange(0.10, 0.91, 0.01)

utilities = []
for t in thresholds:
    pred = val_proba >= t
    tp = int(((pred == 1) & (val_y == 1)).sum())
    fp = int(((pred == 1) & (val_y == 0)).sum())
    # TP reward (INR 600) minus outreach cost (TP+FP) * INR 80
    utility = (tp * 600) - ((tp + fp) * 80)
    utilities.append(utility)

best_threshold = float(thresholds[np.argmax(utilities)])
print(f"Optimized decision threshold: {best_threshold:.4f} (Max validation utility: INR {max(utilities):,})")

# Evaluate on test set
test_y = y[splits["test"]]
test_proba = best_model.predict_proba(X[splits["test"]])[:, 1]
test_eval = evaluate_model(test_y, test_proba, best_threshold)
print("Test evaluation at optimized threshold:")
print(json.dumps(test_eval, indent=2))


Selected model: baseline_logistic_regression


Optimized decision threshold: 0.2000 (Max validation utility: INR 64,840)
Test evaluation at optimized threshold:
{
  "threshold": 0.2,
  "accuracy": 0.6845,
  "precision": 0.6183,
  "recall": 0.9643,
  "f1": 0.7535,
  "roc_auc": 0.885,
  "pr_auc": 0.8784,
  "confusion_matrix": [
    [
      68,
      100
    ],
    [
      6,
      162
    ]
  ]
}


## 6. Feature Importances

In [7]:
feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
clf = best_model.named_steps["model"]
if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
else:
    importances = np.abs(clf.coef_[0])

fi_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
print(fi_df.head(15))


                              feature  importance
0                   num__recency_days    1.744016
2                  num__monetary_180d    0.440689
37  cat__preferred_category_Fragrance    0.384134
30   cat__acquisition_channel_Organic    0.378022
3               num__return_rate_180d    0.341913
33         cat__loyalty_tier_Platinum    0.315393
7               num__ticket_count_90d    0.301747
8       num__negative_ticket_rate_90d    0.300531
18           num__last_visit_days_ago    0.293952
4          num__avg_discount_pct_180d    0.290645
36  cat__preferred_category_Baby Care    0.282206
11                  num__sessions_30d    0.204736
40  cat__preferred_category_Skin Care    0.190011
17           num__campaign_clicks_30d    0.177054
1                 num__frequency_180d    0.168884


## 7. Error Analysis
Examine false positives and false negatives to diagnose weaknesses in the model.

In [8]:
val_test_df = df[splits["validation"] | splits["test"]].copy()
all_proba = best_model.predict_proba(X)[:, 1]
val_test_df["predicted_probability"] = all_proba[splits["validation"] | splits["test"]]
val_test_df["predicted_class"] = (val_test_df["predicted_probability"] >= best_threshold).astype(int)

# False Positives (predicted churn, did not churn)
fp = val_test_df[(val_test_df.predicted_class == 1) & (val_test_df.churn_next_60d == 0)].sort_values("predicted_probability", ascending=False).head(5)
print("=== False Positives (Top 5) ===")
for _, r in fp.iterrows():
    print(f"Customer {r.customer_id}: predicted {r.predicted_probability:.1%} churn, actual no-churn. recency: {r.recency_days}d, sessions: {r.sessions_30d}, tickets: {r.ticket_count_90d}, monetary: INR {r.monetary_180d:.0f}")

# False Negatives (predicted no-churn, churned)
fn = val_test_df[(val_test_df.predicted_class == 0) & (val_test_df.churn_next_60d == 1)].sort_values("predicted_probability", ascending=True).head(5)
print("\n=== False Negatives (Top 5) ===")
for _, r in fn.iterrows():
    print(f"Customer {r.customer_id}: predicted {r.predicted_probability:.1%} churn, actual churn. recency: {r.recency_days}d, sessions: {r.sessions_30d}, tickets: {r.ticket_count_90d}, monetary: INR {r.monetary_180d:.0f}")


=== False Positives (Top 5) ===
Customer CUST01246: predicted 98.5% churn, actual no-churn. recency: 262d, sessions: 1, tickets: 0, monetary: INR 0
Customer CUST01864: predicted 97.6% churn, actual no-churn. recency: 221d, sessions: 3, tickets: 0, monetary: INR 0
Customer CUST01325: predicted 96.3% churn, actual no-churn. recency: 186d, sessions: 1, tickets: 0, monetary: INR 0
Customer CUST02313: predicted 95.1% churn, actual no-churn. recency: 171d, sessions: 3, tickets: 0, monetary: INR 583
Customer CUST01411: predicted 94.6% churn, actual no-churn. recency: 183d, sessions: 0, tickets: 0, monetary: INR 0

=== False Negatives (Top 5) ===
Customer CUST02072: predicted 5.2% churn, actual churn. recency: 35d, sessions: 4, tickets: 0, monetary: INR 4340
Customer CUST01700: predicted 7.1% churn, actual churn. recency: 6d, sessions: 14, tickets: 0, monetary: INR 2045
Customer CUST00184: predicted 7.4% churn, actual churn. recency: 14d, sessions: 6, tickets: 0, monetary: INR 2457
Customer CU

## 8. Save Model Bundle

In [9]:
bundle = {
    "model": best_model,
    "threshold": best_threshold,
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "selected_model": best_name
}
joblib.dump(bundle, "model.pkl")
print("Model bundle successfully saved to model.pkl!")


Model bundle successfully saved to model.pkl!
